# Solutions · Chapter 05-09 · Regularisation

Worked answers to every exercise in `notebooks/05_regression/05-09_regularisation.ipynb`.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# SYNTHETIC: the chapter's 200 rows and 20 columns, of which three matter.
# TRUTH: col 0 = +3.0, col 5 = -2.0, col 9 = +1.5, all others 0, noise sd 1.0.
# Column 1 is a deliberate near-copy of column 0.
wide_rng = np.random.default_rng(17)
n_wide = 200
wide_X = wide_rng.normal(size=(n_wide, 20))
wide_X[:, 1] = wide_X[:, 0] + wide_rng.normal(0, 0.15, n_wide)
wide_truth = np.zeros(20)
wide_truth[0], wide_truth[5], wide_truth[9] = 3.0, -2.0, 1.5
wide_y = wide_X @ wide_truth + wide_rng.normal(0, 1.0, n_wide)

train_X, test_X, train_y, test_y = train_test_split(wide_X, wide_y, test_size=0.3,
                                                    random_state=0)
scaler = StandardScaler().fit(train_X)
scaled_train, scaled_test = scaler.transform(train_X), scaler.transform(test_X)
folds = KFold(5, shuffle=True, random_state=0)


def test_rmse(model, X=scaled_test, y=test_y):
    return float(np.sqrt(((y - model.predict(X)) ** 2).mean()))


print("%d train rows, %d test rows, 20 columns, 3 of them real" % (len(train_X), len(test_X)))

## Quick understanding

### E1 · The two penalties

$$\text{ridge: } \alpha \sum_j w_j^2 \qquad\qquad \text{lasso: } \alpha \sum_j |w_j|$$

**To a coefficient that is already very small**, ridge does almost nothing: the penalty it charges is
`w²`, which at `w = 0.001` is a millionth, and the *pressure* it applies is the derivative `2w`, which is
two thousandths. There is essentially no reason left to move it.

**Lasso applies the same pressure to a tiny coefficient as to a large one.** The derivative of `|w|` is
`±1` everywhere except at zero itself. So a coefficient whose contribution to the fit is worth less than
that constant pressure gets pushed to exactly zero and pinned there.

### E2 · Why scaling is required

**Least squares is scale-invariant.** Multiply a column by 1,000 and its coefficient divides by 1,000;
the product, and therefore every prediction, is unchanged. The chapter measured it: the largest change in
any prediction was **0.0000**.

**A penalty is charged on the coefficient, and the coefficient depends on the units.** Inflating a column
by 1,000 divides its coefficient by 1,000 and its ridge penalty by a *million*, so that column becomes
effectively unpenalised and the burden shifts onto the others. Ridge's predictions moved by **0.6353**
from the same change.

> **Without scaling, `alpha` means a different thing for every column, and the meaning is set by
> whoever chose the units.**

### E3 · One situation for each

**Ridge** when the features are correlated and you want the model to be stable - for example, several
sensors measuring related quantities, or a set of macroeconomic indicators. Ridge shares the coefficient
between them rather than picking one, so refitting on new data does not produce a different story.

**Lasso** when you believe most of the features are irrelevant and the model has to be short - for
example, 4,000 gene-expression columns where a handful matter, or a model that a clinician has to read
and act on. The zeros are the deliverable.

## Hand calculation

### E4 · Ridge shrinkage

`w = 4 / (1 + alpha)`:

| alpha | 0 | 1 | 3 | 9 |
|---|---|---|---|---|
| **ridge w** | **4.0** | **2.0** | **1.0** | **0.4** |

**Multiplicative, and never zero.** Each row is the previous one divided by something; you can make the
coefficient as small as you like and it will never arrive. That is the geometry that makes ridge a
shrinker rather than a selector.

### E5 · Lasso shrinkage

`w = sign(4) * max(|4| - alpha, 0)`:

| alpha | 0 | 1 | 3 | 4 | 9 |
|---|---|---|---|---|---|
| **lasso w** | **4.0** | **3.0** | **1.0** | **0.0** | **0.0** |

**Subtractive, and it hits zero and stops.** This operation is called **soft thresholding**, and the last
two columns are what is different: at `alpha = 4` the coefficient reaches exactly zero, and at `alpha = 9`
it does not go negative - the `max(..., 0)` holds it there.

**Compare the two tables at alpha = 3.** Both give 1.0, by completely different routes: ridge divided by
4, lasso subtracted 3. Push a little further and ridge gives 0.4 while lasso gives 0.

### E6 · Two identical columns

A single copy would take a coefficient of 6, and the two together must sum to 6 to fit equally well.

| | `(6, 0)` | `(3, 3)` |
|---|---|---|
| **ridge penalty** `w₀² + w₁²` | 36 + 0 = **36** | 9 + 9 = **18** |
| **lasso penalty** `|w₀| + |w₁|` | 6 + 0 = **6** | 3 + 3 = **6** |

**Ridge will split them, because 18 is cheaper than 36.** In general, spreading a fixed total across `k`
identical columns divides the ridge penalty by `k`, so ridge shares equally - the chapter's +1.651 and
+1.148.

**Lasso is exactly indifferent**, because the sum of absolute values is 6 for any split. The tie is broken
by whatever the fit marginally prefers on these particular rows - which is why lasso produces one column
and a zero, and why that choice deserves less confidence than it looks.

### E7 · Coefficients 0.001, 0.002 and 40.0

**I suspect the columns were not scaled.**

A three-orders-of-magnitude spread in a penalised model's coefficients almost always means a
three-orders-of-magnitude spread in the columns' units, not in their importance. And it has a
consequence: the two tiny coefficients are being penalised almost not at all while the 40.0 is absorbing
the entire penalty - or the reverse, depending on which way the units run.

**What I would check first: `X.std(axis=0)`.** If those numbers span orders of magnitude, that is the
answer and the fix is a `StandardScaler` inside the pipeline.

**If the columns *are* scaled**, then the reading is different and more interesting: a coefficient of 40
on a standardised column means one standard deviation of that feature moves the target by 40 units, which
is either a genuinely dominant feature or - more often - leakage. 04-05's checks apply.

## Coding

### E8 · The pipeline, and what leaking the scaler costs

In [ ]:
alpha_grid = {"ridge__alpha": np.logspace(-2, 3, 30)}

honest = GridSearchCV(Pipeline([("scale", StandardScaler()), ("ridge", Ridge())]),
                      alpha_grid, cv=folds,
                      scoring="neg_root_mean_squared_error").fit(train_X, train_y)
print("scaler inside the pipeline : alpha %.4f   CV RMSE %.4f   test RMSE %.4f"
      % (honest.best_params_["ridge__alpha"], -honest.best_score_,
         float(np.sqrt(((test_y - honest.predict(test_X)) ** 2).mean()))))

leaky_scaler = StandardScaler().fit(np.vstack([train_X, test_X]))   # fitted on everything
leaky = GridSearchCV(Ridge(), {"alpha": np.logspace(-2, 3, 30)}, cv=folds,
                     scoring="neg_root_mean_squared_error").fit(
    leaky_scaler.transform(train_X), train_y)
print("scaler fitted on everything: alpha %.4f   CV RMSE %.4f   test RMSE %.4f"
      % (leaky.best_params_["alpha"], -leaky.best_score_,
         float(np.sqrt(((test_y - leaky.predict(leaky_scaler.transform(test_X))) ** 2).mean()))))

**The leak is worth almost nothing here - CV RMSE 1.1452 against 1.1457 - and that is the honest result,
not a disappointment.**

`StandardScaler` estimates two numbers per column, a mean and a standard deviation, from 140 rows. Adding
60 more rows drawn from the same distribution barely moves them, so the information that leaks is
genuinely tiny.

**That is a fact about `StandardScaler`, not about leakage.** 04-05 measured transforms where the same
mistake is catastrophic - target encoding, where the leaked quantity is the target itself, and
imputation, where a missing value is filled from rows the model will be scored on. The size of a leak is
the size of what leaked.

**So why insist on the pipeline?** Three reasons that survive this measurement:

1. **You cannot tell in advance which transform you are dealing with.** The habit costs nothing and the
   exceptions are expensive.
2. **The pipeline is what you deploy.** Fitting the scaler outside means the serving code has to
   reproduce it, which is where the mismatch actually happens.
3. **It composes.** A pipeline with a scaler is harmless; the same pipeline with a scaler *and* an imputer
   *and* an encoder is not, and there is no point at which you would go back and add it.

### E9 · Which column survives longest

In [ ]:
exit_alpha = {}
for alpha in np.logspace(-3, 0.8, 300):
    coefficients = Lasso(alpha=alpha).fit(scaled_train, train_y).coef_
    for column in np.flatnonzero(np.abs(coefficients) > 1e-8):
        exit_alpha[column] = alpha

survival = pd.DataFrame([
    {"column": column, "leaves the model at alpha": alpha,
     "true coefficient": wide_truth[column]}
    for column, alpha in sorted(exit_alpha.items(), key=lambda pair: -pair[1])])
print(survival.head(8).to_string(index=False, float_format=lambda v: "%.4f" % v))

**Column 0 survives longest, to `alpha = 2.7005`, then column 5 at 1.5040, then column 9 at 1.2618. The
first noise column drops out at 0.2056 - six times earlier.**

**The survival order is the order of the true coefficients: 3.0, 2.0, 1.5.** That is not a coincidence
and it is the useful part of a lasso path: the order in which columns leave ranks them by how much
evidence supports them, and the gap between the last real column (1.2618) and the first noise column
(0.2056) is a **factor of six** - a visible cliff.

**But read that cliff carefully.** It exists here because the three real effects are large relative to
the noise. Halve them and the cliff would be a slope, and the ordering near the boundary would be
uninformative. The path shows you the ranking; it does not tell you where to cut.

### E10 · The one-standard-error rule as a function

In [ ]:
def one_standard_error_choice(estimator_for, alphas, X, y, cv):
    means, errors = [], []
    for alpha in alphas:
        scores = -cross_val_score(estimator_for(alpha), X, y, cv=cv,
                                  scoring="neg_root_mean_squared_error")
        means.append(scores.mean())
        errors.append(scores.std(ddof=1) / np.sqrt(len(scores)))
    means, errors = np.array(means), np.array(errors)
    best = int(np.argmin(means))
    within = np.flatnonzero(means <= means[best] + errors[best])
    return {"best alpha": alphas[best], "best CV": means[best],
            "standard error": errors[best],
            "one-SE alpha": alphas[int(np.max(within))],
            "one-SE CV": means[int(np.max(within))]}


ridge_alphas = np.logspace(-2, 3, 40)
choice = one_standard_error_choice(lambda a: Ridge(alpha=a), ridge_alphas,
                                   scaled_train, train_y, folds)
for key, value in choice.items():
    print("%-16s %.4f" % (key, value))

for label, alpha in [("best", choice["best alpha"]), ("one-SE", choice["one-SE alpha"])]:
    fitted = Ridge(alpha=alpha).fit(scaled_train, train_y)
    print("\n%-7s alpha %8.4f  test RMSE %.4f  largest |coef| %.3f  sum |coef| %.3f"
          % (label, alpha, test_rmse(fitted), np.abs(fitted.coef_).max(),
             np.abs(fitted.coef_).sum()))

**For ridge the rule changes the model but not the story, and that difference from lasso is the point.**

The one-SE alpha is much larger than the best one, so the coefficients are pulled harder towards zero -
but **the number of features does not change, because ridge never removes any.** "Simpler" for ridge
means "smaller coefficients", which is a real form of simplicity and a much less dramatic one than
lasso's 7 columns becoming 3.

**When it is worth applying to ridge:** when the coefficients will be interpreted, since smaller ones are
less likely to be artefacts of correlated columns; and when the model must be stable across refits, since
a heavier penalty reduces variance.

**When it is not:** if you only want the prediction, the one-SE choice on a flat curve is by construction
a slightly worse predictor, and for ridge you get no interpretability windfall in exchange.

### E11 · Elastic net against the two extremes

In [ ]:
contenders = [
    ("ridge", lambda: Ridge(), np.logspace(-2, 3, 30)),
    ("elastic net 0.1", lambda: ElasticNet(l1_ratio=0.1), np.logspace(-3, 1, 30)),
    ("elastic net 0.5", lambda: ElasticNet(l1_ratio=0.5), np.logspace(-3, 1, 30)),
    ("elastic net 0.9", lambda: ElasticNet(l1_ratio=0.9), np.logspace(-3, 1, 30)),
    ("lasso", lambda: Lasso(), np.logspace(-3, 0.5, 30)),
]

rows = []
for label, make, grid in contenders:
    search = GridSearchCV(make(), {"alpha": grid}, cv=folds,
                          scoring="neg_root_mean_squared_error").fit(scaled_train, train_y)
    best = search.best_estimator_
    rows.append({"model": label, "alpha": search.best_params_["alpha"],
                 "test RMSE": test_rmse(best),
                 "non-zero": int((np.abs(best.coef_) > 1e-8).sum()),
                 "col 0": best.coef_[0], "col 1 (the copy)": best.coef_[1]})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: "%.4f" % v))

**Lasso wins on this data - 0.9117 with 8 columns - and elastic net at `l1_ratio = 0.9` is close behind
at 0.9267 with 9.**

That ordering is what the data deserves: seventeen of the twenty columns are pure noise, which is
precisely the situation lasso is built for. Ridge, forced to keep all twenty, ends at 1.0138.

**The `col 1` column is the interesting one, because it shows the mixing working as advertised:**

| | col 0 | col 1 (a near-copy, truly 0) |
|---|---|---|
| ridge | +2.406 | **+0.511** |
| elastic net 0.1 | +2.163 | **+0.736** |
| elastic net 0.5 | +2.016 | **+0.837** |
| elastic net 0.9 | +2.422 | **+0.385** |
| lasso | +2.819 | **0.000** |

**Sliding `l1_ratio` from 0 to 1 slides the behaviour from sharing to picking.** At 0.1 the model is
mostly ridge and shares generously; at 0.9 it is mostly lasso and has nearly extinguished the copy.

**The lesson is not "lasso is best".** It is that this dataset was built with sparse truth and no
correlated *group* worth preserving - only one redundant pair. Build a dataset with five genuinely
correlated real features and elastic net wins, because lasso would keep one of the five and discard four
real effects. **The right choice is a statement about your features, and it is the one thing
cross-validation cannot tell you before you look.**

### E12 · Stability selection for a model that never selects

In [ ]:
resampler = np.random.default_rng(3)
resamples = 200
ridge_signs = np.zeros(20)
ridge_large = np.zeros(20)
threshold = 0.25

for _ in range(resamples):
    picked = resampler.integers(0, len(scaled_train), len(scaled_train))
    coefficients = Ridge(alpha=1.0).fit(scaled_train[picked], train_y[picked]).coef_
    ridge_signs += coefficients > 0
    ridge_large += np.abs(coefficients) > threshold

report = pd.DataFrame({
    "column": np.arange(20),
    "true": wide_truth,
    "% positive": 100 * ridge_signs / resamples,
    "% above the threshold": 100 * ridge_large / resamples})
report["sign is consistent"] = (report["% positive"] > 95) | (report["% positive"] < 5)
print(report.sort_values("% above the threshold", ascending=False)
      .head(8).to_string(index=False, float_format=lambda v: "%.1f" % v))
print("\ncolumns whose sign was consistent across resamples: %d of 20"
      % int(report["sign is consistent"].sum()))

**The equivalent question for ridge is not "was it selected" but "was it consistently *anything*".**

Ridge gives every column a non-zero number every time, so counting selections is vacuous. Two
replacements, both computed above:

**How often is the coefficient above a size worth caring about?** Columns 0, 5 and 9 clear 0.25 in
**100%** of resamples; the near-copy column 1 clears it in **79.0%**; the best of the genuine noise
columns manages 22.5%. This is a direct analogue of lasso's selection frequency, with the threshold made
explicit instead of hidden inside `alpha` - which is arguably more honest, since lasso's threshold exists
too and is simply harder to see.

**How often does the coefficient keep its sign?** The three real columns are unanimous - 100%, 100% and
0% positive, the last being column 5, whose true coefficient is negative.

**And now the result that matters, which is not the one this exercise was set up to produce.**

**The sign test passes a pure-noise column.** Column 3 is positive in **95.5%** of resamples and its true
coefficient is exactly zero. By any threshold you would have chosen in advance, it looks like a stable,
real, positive effect. Column 16 is at 93.0% and column 7 at 88.0%.

> **Stability is not correctness.** It measures whether refitting on similar data would change your
> answer - not whether your answer is right. A column that is stably wrong is exactly what you get when
> it correlates with something real by chance, and 20 columns against 140 rows produces several.

Column 1 makes the same point from the other direction: its 79% and 92.5% are *genuinely* stable, because
it is a near-copy of a real cause. **Stability correctly identifies it as carrying signal and cannot tell
you that the signal is borrowed.**

**So use these frequencies to reject, not to accept.** A column that fails them is not worth reporting; a
column that passes has only cleared the lowest bar there is.

## Interpretation

### E13 · The CV curve is flat from alpha 0.01 to 10

**What it tells you: the penalty is not the binding constraint.** Over three orders of magnitude the
model's held-out performance does not care how hard you push, which usually means variance is not what is
limiting you - either there is plenty of data for the number of features, or the error is dominated by
bias and noise that no penalty addresses.

**Which end I would pick: the larger alpha**, for three reasons.

1. **The one-standard-error rule says so.** If they are within noise of each other, take the simpler one.
2. **It is the safer extrapolation.** A more heavily penalised model has smaller coefficients, so it
   makes less extreme predictions on inputs unlike the training data - and production always contains
   some.
3. **It is more stable across refits**, which matters if the model is retrained on a schedule and anyone
   is watching the coefficients.

**And the thing I would do next is not about alpha at all.** A flat curve says regularisation is not
where the improvement is. Draw the learning curve (05-08) and check whether the gap is already closed - if
it is, the answer is more capacity or better features, and no value of alpha will help.

### E14 · "Lasso found the five important features"

**Three questions:**

**1. "How stable is that set?"** Resample the rows 200 times, refit, and count. The chapter's version
kept the three real columns 100% of the time and a pure-noise column 85.5% of the time, averaging 9.5
columns out of 20. A single fit's list is one draw from that distribution.

**2. "Were the columns scaled, and how?"** The penalty is charged in the units of the standardised
column. If a feature was scaled differently from the others - or not at all - its selection says more
about its units than its importance.

**3. "Are any of the five correlated with each other, or with something that was dropped?"** Lasso keeps
one of a redundant pair and discards the rest. The four dropped columns may be just as predictive; they
were merely second in line. Reporting the survivors as "the important variables" quietly asserts that the
discarded ones are not, and lasso never tested that.

**A fourth if the answer to any of those is unsatisfying:** "what happens to the held-out error if you
force those five in and remove everything else?" If a five-feature refit predicts as well as the full
lasso, the claim has some support. If it does not, the other columns were doing work.

## Debugging

### E15 · Regularisation made it worse at every alpha

**Cause one: the columns were not scaled.** This is the first thing to check and the most common. With
unscaled columns the penalty falls almost entirely on the small-scale features, which may be the
informative ones, so every alpha damages the model in the same direction. `X.std(axis=0)` settles it.

**Cause two: variance was not the problem.** Regularisation buys a reduction in variance by accepting
bias. If the model was already underfitting - 05-08's curves converged well above the floor - then there
is no variance to buy and the bias is pure cost. Check the learning curve: a small train-to-validation
gap before you started means the penalty had nothing to do.

**A third worth ruling out:** `alpha` was misread. scikit-learn's `Ridge`, `Lasso` and `ElasticNet` do not
scale the penalty identically, and `LogisticRegression`'s `C` is the *inverse*. A grid that looks
reasonable for one is sometimes absurd for another.

### E16 · `LassoCV` picks alpha at the edge of the grid

**It means the grid was too narrow, and the search hit the wall rather than a minimum.** The reported
value is not the best alpha; it is the best alpha *available*, which is a different thing.

**What to do: extend the grid in that direction and refit.** If it picks the edge again, extend again,
until an interior minimum appears.

**And read what each edge implies while you are there.**

- **The smallest alpha wins:** the data wants little or no regularisation. Confirm by fitting plain least
  squares - if that is as good, say so rather than shipping a penalised model that is pretending to do
  something.
- **The largest alpha wins:** the data wants a very heavy penalty, and if the largest value drives every
  coefficient to zero, the model is telling you that predicting the mean is competitive. That is 04-02's
  baseline, arrived at the hard way, and it is worth checking directly.

**Prefer `np.logspace` over `np.linspace` for these grids.** Alpha acts multiplicatively, so a linear grid
spends most of its points in a region where nothing changes.

## Exam and interview reasoning

### E17 · "What is the difference between L1 and L2 regularisation?"

> "Both add a penalty on coefficient size to the loss, to reduce variance at the cost of some bias. L2 -
> ridge - charges the sum of squares, which shrinks every coefficient towards zero without ever reaching
> it, and when features are correlated it spreads the coefficient between them. L1 - lasso - charges the
> sum of absolute values, which drives some coefficients to exactly zero, so it does feature selection as
> part of fitting, and with correlated features it keeps one and discards the rest. In practice: ridge if
> I think everything contributes a little, lasso if I think most things contribute nothing, elastic net if
> both. And with either, the columns must be scaled, because the penalty is charged in the coefficient's
> units."

**"Why does L1 give exact zeros?"**

> "Because the pressure it applies does not fade as the coefficient shrinks. The derivative of `w²` is
> `2w`, which goes to zero as `w` does - so ridge stops pushing exactly when it would need to push hardest
> to finish the job. The derivative of `|w|` is plus or minus one all the way down, so any feature whose
> contribution to the fit is worth less than that constant gets driven to zero and held there.
>
> Geometrically, the L1 constraint region is a diamond with corners on the axes, and a corner is where
> a coefficient is zero. The contours of the loss touch a pointed region at its point far more often than
> they touch a round one at any particular place - which is the same fact stated in pictures."

**What is being tested:** the first answer should reach "selection versus shrinkage" and "correlated
features" without prompting. The follow-up is checking whether you know the mechanism or have memorised
the diamond picture - the derivative argument is the one that generalises.

## Transfer to a different situation

### E18 · 300 rows and 4,000 features

**What changes: least squares does not exist.** With `p` far larger than `n` the design matrix has rank
at most 300, infinitely many coefficient vectors fit the training data perfectly, and `LinearRegression`
returns an arbitrary one of them. **Regularisation is not an improvement here - it is what makes the
problem well-posed at all**, which E20 shows algebraically.

**What I would fit:**

- **Elastic net, cross-validated over both `alpha` and `l1_ratio`.** Genes come in correlated pathways, so
  pure lasso would keep one gene per pathway and discard the rest of a real biological signal; the ridge
  component keeps the group together.
- **Nested cross-validation** (04-07), because with 300 rows the difference between "choosing on the folds"
  and "scoring on the folds" is large.
- **Repeated CV**, because a single 5-fold split of 300 rows is a lottery.

**What I would refuse to claim:**

**That the selected genes are the causal ones.** With 4,000 columns and 300 rows, many will correlate with
the target by chance, and lasso will keep some of them - the chapter kept a pure-noise column 85.5% of
the time with only 20 columns to choose from. I would report selection frequencies from resampling, not
a list.

**That the coefficient sizes are effect sizes.** Lasso's survivors are biased towards zero by
construction, and which of a correlated group survived was close to arbitrary.

**That held-out performance validates the biology.** A model can predict well from a proxy. The honest
deliverable is: this model predicts at this accuracy; these genes were selected at these frequencies;
here is a shortlist worth an experiment.

**And one thing I would do first:** check whether 300 rows can support any conclusion at all, by running
the whole pipeline on a permuted target. If the permuted version also finds "signal", the procedure is
what is producing it.

## Explain it to someone non-technical

### E19 · What a penalty on coefficient size is doing

> The model works by giving each piece of information a weight. Left alone, it will happily give one
> thing an enormous positive weight and another an enormous negative one, because on the data we showed
> it those two cancel out and the answer comes out right.
>
> The trouble is that they only cancel on *that* data. Show it next month's and the balance shifts and
> the prediction swings wildly.
>
> So we charge the model for large weights. It has to justify them by fitting the data noticeably better,
> and mostly it cannot, so it settles for a set of modest weights that survives contact with new data.

*(97 words.)* The cancelling pair is the part that earns its place: it is literally the mechanism 05-07
measured - coefficients of 8,331 with alternating signs - and it makes the fix sound like the obvious
response rather than a mathematical trick.

## Optional challenge

### E20 · Ridge as least squares on augmented data

Ridge minimises

$$\|Xw - y\|^2 + \alpha\|w\|^2$$

Now stack $\sqrt{\alpha}\,I$ underneath $X$ and $p$ zeros underneath $y$:

$$\tilde{X} = \begin{pmatrix} X \\ \sqrt{\alpha}\,I \end{pmatrix}, \qquad
\tilde{y} = \begin{pmatrix} y \\ 0 \end{pmatrix}$$

Ordinary least squares on the augmented pair minimises

$$\|\tilde{X}w - \tilde{y}\|^2 = \|Xw - y\|^2 + \|\sqrt{\alpha}\,I w - 0\|^2 = \|Xw - y\|^2 + \alpha\|w\|^2$$

which is the ridge objective exactly.

In [ ]:
check_rng = np.random.default_rng(2)
rows_here, columns_here, alpha_here = 80, 6, 2.5
X_check = check_rng.normal(size=(rows_here, columns_here))
y_check = X_check @ check_rng.normal(size=columns_here) + check_rng.normal(0, 1, rows_here)

ridge_direct = Ridge(alpha=alpha_here, fit_intercept=False).fit(X_check, y_check)

augmented_X = np.vstack([X_check, np.sqrt(alpha_here) * np.eye(columns_here)])
augmented_y = np.concatenate([y_check, np.zeros(columns_here)])
ridge_by_ols = LinearRegression(fit_intercept=False).fit(augmented_X, augmented_y)

print("ridge directly    :", np.round(ridge_direct.coef_, 8))
print("OLS on augmented  :", np.round(ridge_by_ols.coef_, 8))
print("largest difference: %.2e" % np.abs(ridge_direct.coef_ - ridge_by_ols.coef_).max())

In [ ]:
# a design that is exactly singular: one column repeated
singular_X = np.column_stack([X_check, X_check[:, 0]])

print("condition number of X.T @ X          : %.3e" % np.linalg.cond(singular_X.T @ singular_X))
print("condition number of X.T @ X + alpha I: %.3e"
      % np.linalg.cond(singular_X.T @ singular_X
                       + alpha_here * np.eye(singular_X.shape[1])))

**The two agree to 6.7e-16 - the same computation, written twice.**

**What the identity explains is why ridge is always solvable.** Least squares needs `X.T @ X` to be
invertible; ridge solves `(X.T @ X + alpha I) w = X.T @ y`, and adding `alpha` to every eigenvalue makes
the matrix positive definite for any `alpha > 0`, however degenerate `X` is.

**The numbers make it concrete.** On a design with one column repeated exactly, `X.T @ X` has a condition
number of **2.22e+16** - numerically singular, and least squares returns whatever the solver's
pseudo-inverse happens to produce. Adding `alpha = 2.5` to the diagonal brings it to **54.1**, an ordinary
well-behaved matrix.

**Three readings, all useful.**

The augmentation says what ridge *believes*: it is least squares on your data plus `p` invented rows, each
saying "this coefficient is zero", with a weight of `sqrt(alpha)`. **The penalty is a prior, expressed as
fictional data.**

It explains the `p >> n` case from E18. With 4,000 columns and 300 rows, least squares has no unique
answer; ridge appends 4,000 rows of evidence and does.

And it connects back to 05-06: the condition number governs how hard the problem is to solve, and ridge
improves it **by construction**. A ridge penalty is the cheapest available fix for a badly conditioned
design - and 05-07's degree-12 expansion, at 7.5e+14, was exactly such a design.

### E21 · Making them disagree

In [ ]:
def compare_on(design_seed, kind):
    rng = np.random.default_rng(design_seed)
    n_rows, n_columns = 200, 30
    X = rng.normal(size=(n_rows, n_columns))
    truth = np.zeros(n_columns)
    if kind == "sparse":
        # three big effects, everything else exactly zero: lasso's world
        truth[[0, 1, 2]] = [4.0, -3.0, 2.5]
    else:
        # every column contributes a little: ridge's world
        truth[:] = rng.normal(0, 0.8, n_columns)
    y = X @ truth + rng.normal(0, 1.0, n_rows)

    X_a, X_b, y_a, y_b = train_test_split(X, y, test_size=0.3, random_state=0)
    scale = StandardScaler().fit(X_a)
    A, B = scale.transform(X_a), scale.transform(X_b)

    out = {"data": kind}
    for label, make, grid in [("ridge", lambda: Ridge(), np.logspace(-2, 3, 30)),
                              ("lasso", lambda: Lasso(), np.logspace(-3, 0.5, 30))]:
        search = GridSearchCV(make(), {"alpha": grid}, cv=folds,
                              scoring="neg_root_mean_squared_error").fit(A, y_a)
        out[label] = float(np.sqrt(((y_b - search.best_estimator_.predict(B)) ** 2).mean()))
    out["winner"] = "lasso" if out["lasso"] < out["ridge"] else "ridge"
    return out


print(pd.DataFrame([compare_on(5, "sparse"), compare_on(5, "dense")])
      .to_string(index=False, float_format=lambda v: "%.4f" % v))

**The property that decides it is whether the truth is sparse.**

On the **sparse** design - three large effects and twenty-seven exact zeros - lasso wins, because setting
twenty-seven coefficients to zero is not an approximation, it is the correct answer, and every non-zero
value ridge assigns them is pure variance.

On the **dense** design - thirty small effects, none of them zero - **ridge wins**, because there is
nothing to remove. Lasso's zeros are now errors: each one discards a real contribution in exchange for a
little less variance.

**Be honest about the sizes, though.** Lasso beats ridge by **0.097** on the sparse design and ridge beats
lasso by **0.0044** on the dense one - a margin twenty times smaller, and well inside what a different
split would move. **The sparse case is a real result; the dense case is a directional one.** That
asymmetry is itself informative: choosing lasso when the truth is dense costs little, while choosing
ridge when the truth is sparse costs a lot, so lasso is the less risky default when you genuinely do not
know.

> **Neither penalty is better. Each encodes a belief about the coefficients, and it wins when the belief
> is true.** Ridge assumes many small effects; lasso assumes a few large ones and many exact zeros.

**Which is why the honest workflow is to cross-validate over `l1_ratio` as well as `alpha` when you have
no strong belief** - and why, when you do have one, it is worth stating out loud, because it is an
assumption about the world that the data will not fully test.

**One caveat about this demonstration.** Both designs here have independent columns, chosen so that the
sparsity is the only thing that varies - the module 04 discipline of changing one thing at a time. With
correlated columns a third outcome appears, and it is common in practice: elastic net beats both, because
the truth is sparse in *groups* rather than in individual columns.